# Preprocessing Benchmark — 5 modes

Compare the 5 preprocessing modes on a sample of RSNA DICOM images:
- `raw` — direct load, normalize [0,1]
- `crop_only` — ROI crop only
- `window_only` — adaptive windowing only
- `full` — crop + windowing (recommended)
- `full_iso` — full + isotropic resampling

Outputs saved to `/kaggle/working/results/`

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Paths (flexible: local or Kaggle) ────────────────────────────────────────
DATA_DIR = os.environ.get('DATA_DIR', '/kaggle/input/rsna-breast-cancer-detection')
CODE_DIR = os.environ.get('CODE_DIR', '/kaggle/input/rsna-breast-cancer-code')
OUT_DIR  = os.environ.get('OUT_DIR',  '/kaggle/working/results')

sys.path.insert(0, CODE_DIR)
os.makedirs(f'{OUT_DIR}/metrics', exist_ok=True)
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)

print(f'DATA_DIR : {DATA_DIR}')
print(f'CODE_DIR : {CODE_DIR}')

In [ ]:
# ── Load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(f'{DATA_DIR}/train.csv')
print(f'{len(df)} images — {df.cancer.mean()*100:.2f}% cancer')
df['density'] = df['density'].fillna('B')
df.head(3)

In [ ]:
# ── Init pipeline for each mode ───────────────────────────────────────────────
from core.configuration import Config
from preprocess.pipeline import PreprocessPipeline

IMAGES_DIR = f'{DATA_DIR}/train_images'
config = Config(csv_path=f'{DATA_DIR}/train.csv', images_dir=IMAGES_DIR)

MODES = ['raw', 'crop_only', 'window_only', 'full']
pipelines = {m: PreprocessPipeline(config, mode=m, target_hw=(512, 256)) for m in MODES}
print('Pipelines initialisés :', list(pipelines.keys()))

In [ ]:
# ── Sample: 4 images (2 cancer, 2 non-cancer, densités variées) ───────────────
N_SAMPLE = 8
sample = pd.concat([
    df[df.cancer == 1].sample(min(4, (df.cancer==1).sum()), random_state=42),
    df[df.cancer == 0].sample(min(4, (df.cancer==0).sum()), random_state=42)
]).reset_index(drop=True)

def get_dicom_path(row):
    return f"{IMAGES_DIR}/{row['patient_id']}/{row['image_id']}.dcm"

sample['dicom_path'] = sample.apply(get_dicom_path, axis=1)
sample = sample[sample['dicom_path'].apply(os.path.exists)].head(N_SAMPLE).reset_index(drop=True)
print(f'{len(sample)} images disponibles pour le benchmark')
sample[['patient_id','image_id','laterality','view','density','cancer']]

In [ ]:
# ── Benchmark timing + collecte images ───────────────────────────────────────
results = {m: {'times': [], 'images': []} for m in MODES}

for _, row in sample.iterrows():
    pid      = int(row['patient_id'])
    iid      = int(row['image_id'])
    lat      = str(row['laterality'])
    view     = str(row['view'])
    density  = str(row['density'])
    dpath    = row['dicom_path']

    for mode, pipe in pipelines.items():
        t0 = time.time()
        try:
            img = pipe.process_one(pid, iid, lat, view, density, dpath)
            results[mode]['times'].append(time.time() - t0)
            results[mode]['images'].append(img)
        except Exception as e:
            print(f'  [{mode}] ERREUR {pid}/{iid}: {e}')
            results[mode]['times'].append(None)
            results[mode]['images'].append(np.zeros((512, 256), dtype=np.float32))

# Timing summary
timing = {}
for mode in MODES:
    times = [t for t in results[mode]['times'] if t is not None]
    timing[mode] = {'mean_s': round(np.mean(times), 3), 'std_s': round(np.std(times), 3)}
    print(f'  {mode:15s}: {timing[mode]["mean_s"]:.3f}s ± {timing[mode]["std_s"]:.3f}s')

In [ ]:
# ── Visualisation — grille modes × images ─────────────────────────────────────
n_show = min(4, len(sample))
fig, axes = plt.subplots(len(MODES), n_show, figsize=(4*n_show, 4*len(MODES)))
fig.suptitle('Preprocessing modes comparison', fontsize=16, fontweight='bold', y=1.01)

for r, mode in enumerate(MODES):
    for c in range(n_show):
        ax = axes[r][c] if n_show > 1 else axes[r]
        img = results[mode]['images'][c]
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        row = sample.iloc[c]
        label = 'CANCER' if row['cancer'] == 1 else 'normal'
        if r == 0:
            ax.set_title(f'D={row["density"]} {label}\n{row["view"]}', fontsize=9)
        if c == 0:
            ax.set_ylabel(f'{mode}\n{timing[mode]["mean_s"]:.2f}s', fontsize=9, rotation=0, labelpad=60)
        ax.axis('off')

plt.tight_layout()
fig_path = f'{OUT_DIR}/figures/preprocessing_modes.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig_path}')

In [ ]:
# ── Histogrammes d'intensité par mode ─────────────────────────────────────────
fig, axes = plt.subplots(1, len(MODES), figsize=(4*len(MODES), 4))
fig.suptitle('Intensity distributions per mode', fontsize=13)

for ax, mode in zip(axes, MODES):
    for img in results[mode]['images']:
        ax.hist(img.flatten(), bins=100, alpha=0.3, density=True)
    ax.set_title(mode)
    ax.set_xlabel('pixel value')
    ax.set_xlim(0, 1)

plt.tight_layout()
hist_path = f'{OUT_DIR}/figures/preprocessing_histograms.png'
plt.savefig(hist_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sauvegarde métriques JSON ─────────────────────────────────────────────────
metrics = {
    'n_images': len(sample),
    'timing_per_mode': timing,
    'modes_tested': MODES,
    'target_hw': [512, 256]
}

json_path = f'{OUT_DIR}/metrics/preprocessing_benchmark.json'
with open(json_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('Métriques sauvegardées :')
print(json.dumps(metrics, indent=2))